# Create Brain Aneurysm Foundation Awards

**Brain Aneurysm Foundation** (funder_id `4320314312`, IAMHRF expanded, priority `307`).
bafound.org research-grant-awards year accordions 2022-2025 via Playwright (WPE 403 for
plain requests; headless default-UA chromium gets 200). 62 grants, 100% PI/institution/
title/USD amount/year. Pre-2022 verified non-existent at source.
Parquet: `s3://openalex-ingest/awards/brain_aneurysm/brain_aneurysm_grants.parquet`

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.brain_aneurysm_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/brain_aneurysm/brain_aneurysm_grants.parquet`;

In [ ]:
%sql
-- Check row count (should be ~62)
SELECT COUNT(*) as total_projects FROM openalex.awards.brain_aneurysm_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.brain_aneurysm_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.brain_aneurysm_raw LIMIT 5;

## Step 2: Create Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.brain_aneurysm_awards
USING delta
AS
WITH
funder_row AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320314312  -- Brain Aneurysm Foundation
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        TRY_CAST(g.amount AS DECIMAL(18,2)) as amount,
        'USD' as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.sponsor as funder_scheme,
        'brain_aneurysm' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.year_awarded AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        'United States' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.brain_aneurysm_raw g
    CROSS JOIN funder_row f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'brain_aneurysm' AND priority = 307;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    307 as priority
FROM openalex.awards.brain_aneurysm_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.brain_aneurysm_awards;

In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt FROM openalex.awards.brain_aneurysm_awards GROUP BY 1;

In [ ]:
%sql
SELECT
    COUNT(*) as total, COUNT(display_name) as has_title, COUNT(amount) as has_amount,
    COUNT(lead_investigator) as has_pi, COUNT(start_year) as has_year
FROM openalex.awards.brain_aneurysm_awards;

In [ ]:
%sql
SELECT COUNT(*) as in_raw FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'brain_aneurysm' AND priority = 307;